<div dir="rtl"><h2>05 — שערים צפויים: מידע מוקדם נכנס, שני ערכי λ יוצאים</h2>
<p>EXP-005 · עונות פיתוח בלבד · כל הפיצ׳רים מחושבים לפני as_of. מחברת זו קוראת תוצרים חתומים ואינה מאמנת מודל. תוצאות מועמדים ייכנסו רק כטיוטה לאחר D3, לפני QA ובחירת עמית.</p>
</div>

In [ ]:
import os
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "config/exp005.yaml").exists():
    if root == root.parent:
        raise RuntimeError("Run from the project checkout")
    root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
import pandas as pd
import plotly.io as pio
from IPython.display import HTML, display

from ligat.viz import goals_notebook as views
from ligat.viz.presentation import display_table

pio.renderers.default = "plotly_mimetype"
data = views.development_data()


def table(frame, names, **kwargs):
    display(display_table(frame.rename_axis(None), names, columns=list(frame.columns), **kwargs))


def show(figure, name):
    figure.show()


display(HTML('<div dir="rtl">הקלט הקפוא אומת; נטענו רק עונות הפיתוח.</div>'))

<div dir="rtl"><h2>0 — ממתין ל-D3, ל-QA ולהכרעת עמית</h2>
<p>כאן יופיעו הפיצ׳רים שנכנסו, הדיוק, ומה עובר ל-M3. טרם נבחר מודל וטרם נכתבה שכבת λ אמיתית. גם כאשר D3 מסתיים, סימוני כלל 6 נשארים להכרעה אנושית.</p>
</div>

<div dir="rtl"><h2>1 — M2 חוזה שערים לכל קבוצה בבוקר המשחק</h2>
<p>λ הוא ממוצע חיובי של שערים צפויים, לא תוצאה מובטחת ולא הסתברות לניצחון. למשל λ=1.6 אומר 1.6 שערים בממוצע במפגשים חוזרים באותם תנאים; משחק יחיד עדיין יכול להסתיים באפס או בארבעה. M3 מקבל שני λ ובונה מטריצת תוצאות; הסימולציה מתרגמת משחקים לטבלה. התרשים מתאר חוזה נתונים, ללא אומדן או רווח סמך.</p>
</div>

In [ ]:
show(views.chain_figure(), "E2a_chain")

<div dir="rtl"><h2>2.1 — כל משחק תורם שתי שורות, עם יעד משלו</h2>
<p>שורה היא קבוצה במשחק. היעד goals_scored הוא מספר השערים באותו משחק, מתוך matches; goals_for של prematch הוא סכום עבר ואינו היעד. הקלט הוא prematch_v2 הקפוא. ספירת השורות היא מפקד מלא של הנתונים, ולכן אין לה רווח סמך.</p>
</div>

In [ ]:
counts = views.season_counts(data)
table(counts[["season", "rows", "games"]], {"season": "עונה", "rows": "שורות", "games": "משחקים"})
show(views.season_counts_figure(data), "E2a_season_counts")

<div dir="rtl"><h2>2.2 — בכל קיפול, ההיסטוריה קודמת לעונת המבחן</h2>
<p>Walk-forward מאמן על העבר ובודק על העונה הבאה. עונות החימום אינן משמשות ולידציה; יש 15 קיפולים. מודלי העצים ו-GLM מתאמנים לקיפול. Maher מתעדכן לפני כל יום משחקים, וגם בו משתמשים רק בתוצאות שקדמו לתחילת היום. הצבע מתאר תוכנית קבועה, ולא שיעור שדורש רווח סמך.</p>
</div>

In [ ]:
show(views.walk_forward_figure(), "E2a_walk_forward")

<div dir="rtl"><h2>2.3 — זמן מחצית קצר מעדיף את הכדורגל העדכני</h2>
<p>המשקל הוא $w(a;h)=2^{-a/h}$, ובלי דעיכה $w=1$. הטווח הוא $(0,1]$. במודלי קיפול $a=s_{validation}-1-s_{training}$: העונה הקודמת מקבלת משקל 1; שתי עונות מעבר לעונת ייחוס זו מקבלות רבע משקל כש-h=1; ב-Maher $a=\Delta days/365.25$. המשקל אינו מנורמל מחדש. אלה עקומות מתמטיות מדויקות, ללא אי-ודאות מדגמית.</p>
</div>

In [ ]:
show(views.decay_figure(), "E2a_season_decay")
show(views.decay_figure(daily=True), "E2a_daily_decay")

<div dir="rtl"><h2>3.1 — Poisson מתאר ספירת שערים, ו-NLL מודד הפתעה</h2>
<p>למספר שערים שלם $Y\ge0$ ולממוצע $\lambda>0$:</p>
$$P(Y=y\mid\lambda)=e^{-\lambda}\lambda^y/y!,\qquad \ell(y,\lambda)=\lambda-y\log\lambda+\log\Gamma(y+1).$$
<p>הסתברות בטווח [0,1]; NLL בטווח [0,∞), וערך נמוך יותר עדיף. למשל שער אחד עם λ=1.6 נותן NLL≈1.130. מניחים שוויון בין הממוצע לשונות של Poisson; זו הנחה שנבדקת בכיול. Poisson deviance הוא $2[y\log(y/\lambda)-(y-\lambda)]$ (איבר האפס מוגדר אפס); הוא מחסיר קבוע של יעד לצורכי אימון, ולכן אין להשוות אותו ישירות ל-NLL גולמי. רגולריזציה משפיעה על האימון, אך לא מתווספת למדד הוולידציה.</p>
<p>בדוגמה הבאה λ=1.6 נבחר להמחשה בלבד, ולא נאמד או נבחר לפי משחק זה. התוצאה נלקחה ממשחק אמיתי. העמודות הן הסתברויות מדויקות במודל; n=1 משחק להמחשה, בלי הסקה סטטיסטית.</p>
</div>

In [ ]:
figure, example = views.poisson_example(data)
table(
    example,
    {
        "game_id": "מזהה משחק",
        "team": "קבוצה",
        "opponent": "יריבה",
        "season": "עונה",
        "observed": "שערים בפועל",
        "lambda_example": "λ להמחשה",
        "nll_example": "NLL להמחשה",
    },
    precision={"nll_example": 4},
)
show(figure, "E2a_poisson_example")

<div dir="rtl"><h2>3.2 — XGBoost מוסיף עצים שמתקנים את חיזוי השערים</h2>
$$\lambda(x)=\exp\left[b+\sum_{t=1}^{T}\eta f_t(x)\right].$$
<p>כל עץ לומד תיקון קטן משילוב פיצ׳רים, למשל פער Elo וביתיות. האקספוננט מבטיח λ חיובי. count:poisson הוא יעד האימון, hist היא שיטת הבנייה. CatBoost משתמש גם הוא בעצים ובהפסד Poisson, עם טיפול משותף בקטגוריות. stage מקודד כן/לא ברשת; מזהי קבוצה ויריבה נבדקים בנפרד בסוף הסולם, עם ערך ״לא מוכרת״ שנקבע מההיסטוריה בלבד.</p>
</div>

<div dir="rtl"><h2>3.3 — GLM נותן בסיס פשוט, Maher מפריד התקפה והגנה</h2>
$$\lambda_i=\exp(\beta_0+x_i^T\beta),\qquad \lambda_{ij}=\alpha_i\beta_j\gamma^{I(home)}.$$
<p>ב-GLM מקדם חיובי מגדיל את קצב השערים, למשל יתרון בית קבוע באותו מערך פיצ׳רים. ב-Maher α היא חוזק ההתקפה, β היא נטייה לספוג, ו-γ היא מכפלת הביתיות; כולם חיוביים. נדרשת מגבלת זיהוי $\sum\log\alpha=0$. כאן Dixon–Coles הוא קו בסיס Maher עם דעיכה ועדכון יומי; אין תיקון תלויות לתוצאות נמוכות (ρ).</p>
<p>לעולה יש prior קפוא בעוצמה של שישה משחקים, סביב ממוצע ליגת העונה $R$:</p>
$$\frac{k}{2}\sum_{p\in P}\left[(\log\alpha_p-\overline{\log\alpha}_R-a^*)^2+(\log\beta_p-\overline{\log\beta}_R-b^*)^2\right].$$
<p>עוצמת הקנס בפועל היא $k=6\bar y_w w_0$: ממוצע שערי העבר המשוקלל מתרגם את ששת המשחקים לעוצמה, ו-$w_0$ הוא משקל תחילת העונה ביחס לבוקר החיזוי (1 ללא דעיכה). כך גם המידע המוקדם מתיישן עם העונה.</p><p>כשיש כמה עולות, נגזרת ממוצע R קושרת את הקנסות. עולה ללא משחקים אינה חייבת להיות בדיוק a*/b*; תנאי האופטימום הוא $d_p=\sum_{q\in P}d_q/|R|$. הקנס אינו נכלל ב-NLL הוולידציה.</p>
</div>

In [ ]:
prior = views.frozen_prior()
table(
    pd.DataFrame([prior]),
    {
        "a_star": "a* — התקפה יחסית בלוג",
        "b_star": "b* — ספיגה יחסית בלוג",
        "source_sha256": "חתימת מקור",
        "team_seasons": "קבוצות־עונות",
    },
    precision={"a_star": 6, "b_star": 6},
)

<div dir="rtl"><h2>4 — הרשת קבועה מראש: דיוק מול מורכבות ורגולריזציה</h2>
<p>היפר-פרמטר הוא בחירה שנקבעת מחוץ ללמידת המקדמים: כמה עמוק העץ, כמה תיקונים וכמה קנס. הרשת כוללת ערכים פשוטים ומורכבים, והבחירה לפי ולידציה עתידית. הטבלה נגזרת ישירות מה-YAML. ערך None בזמן מחצית פירושו בלי דעיכה. ברירת מחדל של ספרייה היא מועמד עצמאי מלא. הגרפים שישוו צירופים יתווספו ב-E2c; אין כאן ציוני מועמדים.</p>
</div>

In [ ]:
table(
    views.hyperparameter_table(),
    {
        "model": "מודל ושלב",
        "parameter": "פרמטר",
        "values": "ערכים שנקבעו מראש",
        "meaning": "משמעות בכדורגל",
        "rationale": "הנימוק",
    },
)

<div dir="rtl"><h2>5 — הסולם</h2>
<p>ממתין לסבב הבא. בפרקי התוצאות: ממתין ל-D3, ל-QA ולהכרעת עמית; בפרק ההסבר יופיעו כיול, שאריות וביתיות לאחר בחירת המודל.</p>
</div>

<div dir="rtl"><h2>6 — התוצאות</h2>
<p>ממתין לסבב הבא. בפרקי התוצאות: ממתין ל-D3, ל-QA ולהכרעת עמית; בפרק ההסבר יופיעו כיול, שאריות וביתיות לאחר בחירת המודל.</p>
</div>

<div dir="rtl"><h2>7 — להבין את המודל</h2>
<p>ממתין לסבב הבא. בפרקי התוצאות: ממתין ל-D3, ל-QA ולהכרעת עמית; בפרק ההסבר יופיעו כיול, שאריות וביתיות לאחר בחירת המודל.</p>
</div>

<div dir="rtl"><h2>8 — בדיקות</h2>
<p>ממתין לסבב הבא. בפרקי התוצאות: ממתין ל-D3, ל-QA ולהכרעת עמית; בפרק ההסבר יופיעו כיול, שאריות וביתיות לאחר בחירת המודל.</p>
</div>

<div dir="rtl"><h2>9 — ה-λ ל-M3</h2>
<p>ממתין לסבב הבא. בפרקי התוצאות: ממתין ל-D3, ל-QA ולהכרעת עמית; בפרק ההסבר יופיעו כיול, שאריות וביתיות לאחר בחירת המודל.</p>
</div>

<div dir="rtl"><h2>10 — סיכום ומילון</h2>
<p>ממתין לסבב הבא. בפרקי התוצאות: ממתין ל-D3, ל-QA ולהכרעת עמית; בפרק ההסבר יופיעו כיול, שאריות וביתיות לאחר בחירת המודל.</p>
</div>